# Intro

- In the previous module we evaluated our system offline, before anyone used it.
    - We measured search quality with Hit Rate and MRR,
    - and answer quality with cosine similarity and an LLM judge.

- But once we deploy, real people start asking real questions. The offline numbers stop telling the whole story.
    - We don't know how long answers take, what they cost, or whether anyone finds them useful.
    - We need to watch the system while it runs.

- That's monitoring: online evaluation.
    - We collect metrics from the running system and put them on a dashboard.
    - Then we can see how it performs with real traffic.

- For every question that comes in, there's a lot we can capture:
    - The instructions, prompt, and model behind the answer
    - Input and output tokens, and how much the call cost
    - Response time: how long the person waited
    - User feedback: a thumbs up or thumbs down on the answer
    - Relevance: does the answer address the question?

- Collecting and viewing all of this takes three new pieces:
    - a user interface where people ask questions
    - a database to store conversations and feedback
    - a dashboard to visualize the metrics

- We build all three on top of the RAG pipeline from the earlier modules.
    - We don't rebuild the RAG part. We wrap it in a Streamlit app and save every interaction to PostgreSQL.
    - Then we put a dashboard in front of the data. At the end we add Grafana for a more powerful view.

- We focus on **RAG** here. Monitoring an **agent** works almost the same way, so we leave it as **homework**.
    - The [agents module](../01_agentic_rag/) already has the pieces you need to apply these same ideas there.

# Assistant

- Before we monitor anything, we need something to monitor. So we start with a RAG pipeline that answers questions about our courses.
- We won't build it from scratch. We already did that in the earlier modules, and the flow is the same three steps as always.
    - First we search the FAQ for the questions most relevant to the user's question.
    - Then we build a prompt from that question plus the documents we found.
    - Finally we send it to the LLM, which gives us the answer.
- That's the whole pipeline, and we reuse it as-is.

## Creating the assistant

- [`assistant.py`](../src/assistant.py) loads the data and builds the index, then hands both to `RAGBase`.
- We pass our own instructions template here.
- Run the assistant

In [ ]:
%%bash
uv run python ../src/assistant.py

In [ ]:
%%bash
uv run python ../src/assistant.py "a different query"

We'll run this command again and again, and typing it in full every time gets old. So we put it in a [`Makefile`](../Makefile).

In [ ]:
%%bash
# sudo apt install make
make -C .. run ARGS='"a different query"'

# Chat App

- The command line works, but we want something closer to how a person would actually talk to the assistant.
- So we wrap it in a small web interface. We use **Streamlit**, a Python framework for building front ends with almost no code.
- This isn't the final product, and it isn't meant to be pretty.

In [ ]:
%%bash
uv add streamlit

- Create and Run [`app.py`](../src/app.py)
    ```bash
    uv run streamlit run ../src/app.py
    ```
- That's another long command, so it goes into the [`Makefile`](../Makefile) too.
- The RAG works, but right now we track nothing about it: no response time, no token usage, no cost.
- That's exactly the visibility monitoring is supposed to give us, so that's what we add next.

In [ ]:
%%bash
make chat # at project root and only terminal works not notebook

# Capturing Metrics

- Create [`metrics.py`](../src/metrics.py).
- Every call produces a bundle of values we want to keep together.
    - We could pass them around as a dictionary, but then you have to remember what keys are in it.
    - A **dataclass** spells the fields out, so anyone reading the code can see what we record for each call.
        - This is the `LLMCallRecord` dataclass

## Cost calculation

- Next we need the cost of each call.
    - The provider charges a price per million input tokens and another per million output tokens.
    - So we multiply each count by its rate and divide by a million.
    - The `usage` object comes straight from the LLM response. It carries the token counts for the call we just made.
    - We define this logic in `calculate_cost` method

## Instrumented RAG

- `RAGBase` already works. So instead of rewriting it, we subclass it and only change the one method that calls the LLM. Everything else stays the same, and every time `rag()` makes a call, the metrics get recorded for free.
- The same trick works for agents. You'd capture each tool call the same way we captured LLM calls in the evaluation module.
- In `metrics.py`, the subclass that captures metrics is `RAGWithMetrics`
    - The `_call_llm` method sends the request to the LLM.
    - The `_log_response` method builds the record and stashes it on `self.last_call`.
        - Keeping state on the object isn't the cleanest design. If two things called it at once, they'd fight over `last_call`.
        - But it lets us read the metrics back without changing the method's return type.
        - For one person clicking through a Streamlit app, that's good enough.
        - We also print the record so we can see what we're capturing. The `_log_response` method captures all the metrics.

## Updating the Streamlit app

- First, update `assistant.py` to import `RAGWithMetrics` instead of `RAGBase`.
- `app.py` doesn't need changes - it still calls `create_assistant()` and `assistant.rag()`.
- But now we can also display the metrics.
- Run the app again:
    ```bash
    make chat
    ```
- Now you can see response time, token usage, and cost under each answer.
- It isn't a pretty panel, but it shows the numbers we care about.
- One naming note: We call these `prompt_tokens` and `completion_tokens` to follow the API. They are actually `input_tokens` and `output_tokens`.
- We capture the metrics now, but they vanish the moment we close the app. Next we save each record to a database so we can track usage over time.

# Storing Data in PostgreSQL

- The metrics disappear when we close the app, so we need somewhere to keep them.
- We store every conversation in PostgreSQL, which we run only for monitoring.
    - No other part of the system touches this database.
- We pick Postgres for two reasons:
    - it handles structured data well,
    - Grafana connects to it easily later on.

## Starting PostgreSQL with Docker

- First we create a Docker network.
    - Postgres and Grafana both run in containers, and Grafana reaches Postgres by name, so they need to share a network.
    ```bash
    docker network create monitoring
    ```
- Start PostgreSQL with a volume for data persistence and connect it to the network:
    ```bash
    docker run -it \
        --name course-assistant-pg \
        --network monitoring \
        -e POSTGRES_USER=user \
        -e POSTGRES_PASSWORD=password \
        -e POSTGRES_DB=course_assistant \
        -p 5432:5432 \
        -v pgdata:/var/lib/postgresql/data \
        postgres:17
    ```

- These are long commands we'll run again and again, so they go in the `Makefile`.
    - The `postgres` target depends on `network`,
    - so `make postgres` creates the network first and then starts the container.

- Now we can just run:
    ```bash
    make postgres
    ```

## Connecting to PostgreSQL

- To reach Postgres from Python, we install the `psycopg` driver:
    ```bash
    uv add "psycopg[binary]"
    ```

## Initializing the database

- The table stores everything from our `LLMCallRecord`, plus the question and the course.
- We call it `conversations`.
- Two fields are worth a word.
    - We store the `course` because the same assistant can serve more than one course. Right now everything is `llm-zoomcamp`, but keeping the column means we don't have to reshape the table when we add others.
    - The `timestamp` is timezone-aware (`TIMESTAMP WITH TIME ZONE`) on purpose.
    - Without the time zone, Grafana won't line the data up correctly on its time axis later.

- The SQL to create the table
- We can run this via `psql` or any other tool, but let's create a Python script.

```sql
CREATE TABLE conversations (
    id SERIAL PRIMARY KEY,
    question TEXT NOT NULL,
    answer TEXT NOT NULL,
    course TEXT NOT NULL,
    model TEXT NOT NULL,
    instructions TEXT NOT NULL,
    prompt TEXT NOT NULL,
    prompt_tokens INTEGER NOT NULL,
    completion_tokens INTEGER NOT NULL,
    total_tokens INTEGER NOT NULL,
    response_time FLOAT NOT NULL,
    cost FLOAT NOT NULL,
    timestamp TIMESTAMP WITH TIME ZONE NOT NULL
)
```

- Create [`db_init.py`](../src/db_init.py).
- A helper to connect to the database. It uses environment variables with defaults matching the Docker container we just started.
- The init function creates the table. With `drop=True` it drops the table first, which wipes all existing data.
    - That's handy while we're still changing the schema and want a clean start.
    - But be careful with it. You never want to run a drop against a real database by accident.
- Run the init script
- We run this once and don't add it to the `Makefile`.
    - The `postgres` container uses a named volume (`pgdata`), so the data survives restarts.
    - The table is still there next time we start Postgres. We only run `db_init.py` again when we change the schema.

In [ ]:
%%bash
uv run python ../src/db_init.py

## Saving conversations

- We want to insert an `LLMCallRecord` into the `conversations` table.
- The `id` column is a `SERIAL`, so Postgres assigns it automatically.
- We add `RETURNING id` to get that value back, because we need it later.
- When a user rates an answer, we have to know which conversation the feedback belongs to.

- The SQL we want to execute:
    ```sql
    INSERT INTO conversations (
        question, answer, course, model, instructions, prompt,
        prompt_tokens, completion_tokens, total_tokens,
        response_time, cost, timestamp
    ) VALUES (
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
    )
    RETURNING id
    ```

- Create [`db_save.py`](../src/db_save.py).
- We pass `question` in separately rather than reading it off the record.
- The record's `prompt` is the full text we send to the model, specific to how we call the LLM.
- The raw question the user typed is a different thing. We want it in its own column so we always know what was actually asked.
- You could fold the question into the prompt instead, but here we keep them apart.
- The save function takes an `LLMCallRecord` and inserts it into the database
    - We can also add it to the `__main__` block in `assistant.py` so every CLI test gets saved.
    - Then add the save call after the answer in the `__main__` block.

- Test it.
    ```bash
    uv run python src/assistant.py "How do I join the course?"
    ```
- Then, check the data.
    ```bash
    docker exec -it course-assistant-pg psql -U user -d course_assistant -c "SELECT id, question, response_time, cost FROM conversations;"
    ```

## Integrating with Streamlit

- In [`app.py`](../src/app.py), just add the import and one line after `assistant.rag()`.
- And later, save the conversation and add the assigned ID to the session state.
- Every question and answer is now saved to PostgreSQL. Next we query the data to pull recent conversations back out.

# Querying Data

- We're saving conversations now, so the next step is reading them back.
- That's what the dashboard runs on.
- Create [`db_query.py`](../src/db_query.py).
- Connect to the same database:
- A query returns each row as a plain tuple. You have to remember that column 4 is the model and column 6 is the prompt.
- That's no fun to work with. So we convert each row back into the `LLMCallRecord` dataclass we already use for live calls.
- A helper to convert a database row into an `LLMCallRecord`.
- Then, update `get_conversations` to use it.
- We order by `timestamp` to get the most recent calls.
- One thing to keep in mind as the table grows:
    - there's no index on `timestamp`, but there is one on `id`.
    - Since ids increase over time anyway, ordering by `id` would be faster - or you add an index on `timestamp`.
    - With a handful of rows it doesn't matter, so we leave it simple for now.

- Test it:
    ```bash
    uv run python src/db_query.py
    ```
- The output is a wall of text, not something you'd want to read all day.
- Still, it proves we can pull the data back out of the database. Next, we put it in front of a **dashboard**.

# Streamlit Dashboard

- Before we reach for Grafana, let's build a quick dashboard right in Streamlit.
- For a lot of projects this is all we need. When you're getting started, seeing latency, cost, and recent conversations in one place is already enough. You often don't need Grafana at all.
- If you stop here, you don't even need Postgres. You could swap it for SQLite and skip Docker entirely. We're on Postgres only because Grafana connects to it more easily than to SQLite, which matters later.
- For a **lightweight** project, **SQLite plus a Streamlit** dashboard is a perfectly good place to stop.
- If you're not a Streamlit expert, describe what you want to ChatGPT or a coding assistant. Then, let it write the layout.

- First, add aggregate queries to [`db_query.py`](../src/db_query.py).
- Add a `Stats` dataclass to [`db_query.py`](../src/db_query.py).
- Add `get_stats` function to compute aggregate stats.
- Then, create [`dashboard.py`](../src/dashboard.py):
- At the top we show four summary numbers, the ones most worth watching when you're getting started.
    - You can show far more, but these are a good starting point.
- For the time charts we pull the last 100 conversations and let Streamlit plot them.
    - This isn't the most efficient way to do it.
    - We fetch whole records just to chart two columns.
    - A leaner version would query only the timestamp and the value we want.
    - With our volume it's fine, so we keep it short.
- We also show recent conversations:

- Run it.
- The port 8501 might be already in use (by the chat app), so we will use a different port:
    ```bash
    uv run streamlit run src/dashboard.py --server.port 8502
    ```
- We didn't even use a table for the conversations - plain text is enough to make the point.
- This simple dashboard already gives us real visibility into the system. Later we set up Grafana for a more powerful view, with alerting and richer panels.